### Option A — ablation: channel LayerNorm + dropout, 3 seeds

Same data, split, targets, integration and metrics as the defended A8q. **Only the
architecture changes**: a channel LayerNorm after each convolution in the TCN blocks
and dropout in the blocks and in the two heads. Trains seeds 1, 2 and 3, selects the
**quat** checkpoint on VAL (exactly as A8q does), and reports mean +/- std TEST
ATE / RTE / quaternion against the frozen A8q (1.12 m / 0.31 m / 4.95 deg),
insfilterMARG and GPS-velocity integration.

The point is to test whether normalization + regularization reduce the seed variance
seen in A9 (1.53 m) relative to A8q (1.12 m), **not** to replace the defended row.

Run with the working directory set to `option_a/` (needs `dataset.csv`,
`filterstate.csv`, `results_a8q.json`). **Does not touch `option_a_a8q.pth` or
`results_a8q.*`.** Writes `ablation_results.json` and `ablation_best.pth`.


In [1]:
import os, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

IMU_FS = 160
WIN_SEC, EXTRACT_SEC, HOP_SEC, RTE_SEC = 1.0, 0.1, 1.0, 1.0
W = int(WIN_SEC * IMU_FS)
STRIDE = int(EXTRACT_SEC * IMU_FS)
HOP_N = int(round(HOP_SEC / EXTRACT_SEC))
BATCH_SIZE, EPOCHS, LR = 256, 80, 1e-3
LAMBDA_ROT = 1.0
VAL_FRAC, TEST_FRAC, GAP = 0.15, 0.15, HOP_N
DROPOUT = 0.1
SEEDS = [1, 2, 3]

assert os.path.exists("dataset.csv"), "run make_dataset.m first (cwd must be option_a/)"
print("ablation LN+dropout", device, "seeds", SEEDS, "dropout", DROPOUT, "W", W)

ablation LN+dropout cuda seeds [1, 2, 3] dropout 0.1 W 160


In [2]:
COLS = [
    "accx","accy","accz","wx","wy","wz",
    "magx","magy","magz","velx","vely","velz",
    "posx","posy","posz","qw","qx","qy","qz",
]
df = pd.read_csv("dataset.csv", names=COLS)
meas = df[COLS[:12]].to_numpy(np.float32)
pos = df[["posx","posy","posz"]].to_numpy(np.float32)
quat = df[["qw","qx","qy","qz"]].to_numpy(np.float32)
print(df.shape)

(22560, 19)


In [3]:
def q_normalize(q, eps=1e-8):
    return q / (np.linalg.norm(q, axis=-1, keepdims=True) + eps)

def q_conj(q):
    o = q.copy(); o[..., 1:] *= -1; return o

def q_mul(a, b):
    aw, ax, ay, az = (a[..., k] for k in range(4))
    bw, bx, by, bz = (b[..., k] for k in range(4))
    return np.stack([
        aw*bw - ax*bx - ay*by - az*bz,
        aw*bx + ax*bw + ay*bz - az*by,
        aw*by - ax*bz + ay*bw + az*bx,
        aw*bz + ax*by - ay*bx + az*bw,
    ], axis=-1)

def q_delta(q0, q1):
    return q_normalize(q_mul(q_conj(q_normalize(q0)), q_normalize(q1)))

def quat_to_rotvec(q):
    q = q_normalize(q)
    q = np.where(q[..., :1] < 0, -q, q)
    w = np.clip(q[..., 0], -1, 1)
    xyz = q[..., 1:]
    n = np.linalg.norm(xyz, axis=-1, keepdims=True)
    angle = 2 * np.arccos(w)
    axis = xyz / np.clip(n, 1e-8, None)
    return (axis * angle[..., None]).astype(np.float32)

def rotvec_to_quat(v, eps=1e-8):
    angle = np.linalg.norm(v, axis=-1, keepdims=True)
    axis = v / np.clip(angle, eps, None)
    half = 0.5 * angle
    return np.concatenate([np.cos(half), axis * np.sin(half)], -1).astype(np.float32)

def q_ang_deg(qa, qb):
    d = np.clip(np.abs((q_normalize(qa) * q_normalize(qb)).sum(-1)), -1, 1)
    return np.degrees(2 * np.arccos(d))

In [4]:
quat = q_normalize(quat)
ends = np.arange(W - 1, len(df), STRIDE)
X = np.stack([meas[t - W + 1:t + 1].T for t in ends], 0)
v_gps = meas[ends, 9:12]
vel = (pos[ends] - pos[ends - (W - 1)]) / WIN_SEC
drot = quat_to_rotvec(q_delta(quat[np.maximum(ends - STRIDE, 0)], quat[ends]))
dv = vel - v_gps
Y = np.concatenate([dv, drot], 1)
pos_end, q_end = pos[ends], quat[ends]
print(X.shape, Y.shape)

(1401, 12, 160) (1401, 6)


In [5]:
N = len(X)
n_test, n_val = int(N * TEST_FRAC), int(N * VAL_FRAC)
n_train = N - n_val - n_test - 2 * GAP
tr = slice(0, n_train)
va = slice(n_train + GAP, n_train + GAP + n_val)
te = slice(n_train + GAP + n_val + GAP, N)
X_tr, Y_tr = X[tr], Y[tr]
X_va, Y_va = X[va], Y[va]
X_te, Y_te = X[te], Y[te]
pos_va, q_va = pos_end[va], q_end[va]
pos_te, q_te = pos_end[te], q_end[te]
v_gps_va, v_gps_te = v_gps[va], v_gps[te]
print(len(X_tr), len(X_va), len(X_te))

961 210 210


In [6]:
feat_mean, feat_std = X_tr.mean((0, 2)), X_tr.std((0, 2)) + 1e-8
y_mean, y_std = Y_tr.mean(0), Y_tr.std(0) + 1e-8
y_mean_t = torch.tensor(y_mean)
y_std_t = torch.tensor(y_std)

def nX(a):
    return (a - feat_mean[None, :, None]) / feat_std[None, :, None]

def nY(a):
    return (a - y_mean) / y_std

def denorm_t(p):
    return p * y_std_t.to(p.device) + y_mean_t.to(p.device)

X_tr_n, X_va_n, X_te_n = nX(X_tr), nX(X_va), nX(X_te)
Y_tr_n = nY(Y_tr)

class IncDS(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

In [7]:
def ate_rte(est, ref, hop=HOP_SEC, sec=RTE_SEC):
    ate = float(np.sqrt(((est - ref) ** 2).sum(1).mean()))
    w = max(int(round(sec / hop)), 1)
    rel = [
        np.linalg.norm((est[i + w] - est[i]) - (ref[i + w] - ref[i]))
        for i in range(0, len(est) - w, max(w // 2, 1))
    ]
    return ate, float(np.mean(rel)) if rel else float("nan")

def integrate_pos(vel, pos0):
    n = len(vel)
    ph = np.zeros((n, 3), np.float32)
    ph[0] = pos0
    for t in range(1, n):
        ph[t] = ph[t - 1] + vel[t] * HOP_SEC
    return ph

def integrate_q(drot, q0):
    n = len(drot)
    qh = np.zeros((n, 4), np.float32)
    qh[0] = q0
    for t in range(1, n):
        qh[t] = q_normalize(q_mul(qh[t - 1], rotvec_to_quat(drot[t])))
    return qh

@torch.no_grad()
def predict_inc(model, Xn):
    model.eval()
    out = []
    for i in range(0, len(Xn), BATCH_SIZE):
        xb = torch.tensor(Xn[i:i + BATCH_SIZE], device=device)
        p = model(xb).cpu() * y_std_t + y_mean_t
        out.append(p.numpy())
    return np.concatenate(out)

def split_ate(model, Xn, vg, pos_ref, q_ref):
    inc = predict_inc(model, Xn)
    v = vg + inc[:, :3]
    ph = integrate_pos(v[::HOP_N], pos_ref[::HOP_N][0])
    qh = integrate_q(inc[:, 3:], q_ref[0])
    ate, _ = ate_rte(ph, pos_ref[::HOP_N])
    return ate, float(q_ang_deg(qh, q_ref).mean())

def score_test(model):
    inc = predict_inc(model, X_te_n)
    v_hat = v_gps_te + inc[:, :3]
    pos_1s = pos_te[::HOP_N]
    pos_h = integrate_pos(v_hat[::HOP_N], pos_1s[0])
    q_h = integrate_q(inc[:, 3:], q_te[0])
    ate, rte = ate_rte(pos_h, pos_1s)
    quat = float(q_ang_deg(q_h, q_te).mean())
    return ate, rte, quat

In [8]:
class ChanLN(nn.Module):
    # LayerNorm over the channel dimension of a (N, C, T) tensor.
    def __init__(self, ch):
        super().__init__()
        self.ln = nn.LayerNorm(ch)
    def forward(self, x):
        return self.ln(x.transpose(1, 2)).transpose(1, 2)

class TcnBlock(nn.Module):
    def __init__(self, ch, k, d, p=0.1):
        super().__init__()
        pad = (k - 1) * d // 2
        self.f = nn.Sequential(
            nn.Conv1d(ch, ch, k, padding=pad, dilation=d), ChanLN(ch), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(ch, ch, k, padding=pad, dilation=d), ChanLN(ch),
        )
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(x + self.f(x))

class IncModelLN(nn.Module):
    def __init__(self, p=DROPOUT):
        super().__init__()
        self.stem = nn.Conv1d(12, 64, 5, padding=2)
        self.tcn = nn.Sequential(
            TcnBlock(64, 5, 1, p),
            TcnBlock(64, 5, 2, p),
            TcnBlock(64, 5, 4, p),
            TcnBlock(64, 5, 8, p),
        )
        self.head_v = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 3))
        self.head_q = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 3))
    def forward(self, x):
        z = torch.relu(self.stem(x))
        z = self.tcn(z)
        h = torch.cat([z.mean(-1), z[:, :, -1]], 1)
        return torch.cat([self.head_v(h), self.head_q(h)], 1)

_m = IncModelLN()
print(_m(torch.zeros(2, 12, W)).shape, "params", sum(p.numel() for p in _m.parameters()))

torch.Size([2, 6]) params 186182


In [9]:
# data-derived baselines (same for every seed)
pos_te_1s = pos_te[::HOP_N]
pos_gps = integrate_pos(v_gps_te[::HOP_N], pos_te_1s[0])
ate_g, rte_g = ate_rte(pos_gps, pos_te_1s)

diag = np.loadtxt("filterstate.csv", delimiter=",")
ekf_pos = diag[:, 1:4].astype(np.float32)
ekf_q = q_normalize(diag[:, 4:8].astype(np.float32))
ends_te = ends[te]
pos_ekf = ekf_pos[ends_te[::HOP_N]]
q_ekf = q_normalize(ekf_q[ends_te])
ate_e, rte_e = ate_rte(pos_ekf, pos_te_1s)
quat_e = float(q_ang_deg(q_ekf, q_te).mean())
print(f"GPS-vel {ate_g:.3f}/{rte_g:.3f}   EKF {ate_e:.3f}/{rte_e:.3f}/{quat_e:.2f}")

GPS-vel 1.103/0.451   EKF 0.775/0.201/7.46


In [10]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def snap(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

rows = []
best_overall = None
for s in SEEDS:
    set_seed(s)
    loader = DataLoader(IncDS(X_tr_n, Y_tr_n), BATCH_SIZE, shuffle=True)
    model = IncModelLN(DROPOUT).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=8, factor=0.5)
    best_q, state_q, rec = float("inf"), None, None
    for e in range(EPOCHS):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = nn.functional.mse_loss(pred[:, :3], yb[:, :3]) + LAMBDA_ROT * nn.functional.mse_loss(pred[:, 3:], yb[:, 3:])
            opt.zero_grad(); loss.backward(); opt.step()
        ate_v, qv = split_ate(model, X_va_n, v_gps_va, pos_va, q_va)
        sched.step(ate_v + qv / 30)
        if qv < best_q:
            best_q, state_q, rec = qv, snap(model), (ate_v, qv)
    model.load_state_dict(state_q)
    ate, rte, quat = score_test(model)
    print(f"seed {s}: val best-quat ATE {rec[0]:.3f} quat {rec[1]:.2f}  ->  TEST ATE {ate:.3f} RTE {rte:.3f} quat {quat:.2f}")
    rows.append({"seed": s, "ate_m": ate, "rte_m": rte, "quat_mean_deg": quat,
                 "val_ate": rec[0], "val_quat": rec[1]})
    if best_overall is None or quat < best_overall[0]:
        best_overall = (quat, snap(model), s)

A = np.array([[r["ate_m"], r["rte_m"], r["quat_mean_deg"]] for r in rows])
mean, std = A.mean(0), A.std(0)
a8q = json.load(open("results_a8q.json"))["quat_ckpt"]

print("\n==== ablation (channel LayerNorm + dropout) vs A8q ====")
print(f"{'':20}{'ATE':>8}{'RTE':>8}{'quat':>8}")
print(f"{'A8q (frozen)':20}{a8q['ate_m']:8.2f}{a8q['rte_m']:8.2f}{a8q['quat_mean_deg']:8.2f}")
print(f"{'ablation mean':20}{mean[0]:8.2f}{mean[1]:8.2f}{mean[2]:8.2f}")
print(f"{'ablation std':20}{std[0]:8.2f}{std[1]:8.2f}{std[2]:8.2f}")
print(f"{'GPS-vel':20}{ate_g:8.2f}{rte_g:8.2f}{'--':>8}")
print(f"{'insfilterMARG':20}{ate_e:8.2f}{rte_e:8.2f}{quat_e:8.2f}")

out = {
    "run_id": "ABLATION_LN_DROPOUT",
    "arch": "dual_head_tcn_channel_layernorm_dropout",
    "dropout": DROPOUT, "seeds": SEEDS, "epochs": EPOCHS,
    "per_seed": rows,
    "mean": {"ate_m": float(mean[0]), "rte_m": float(mean[1]), "quat_mean_deg": float(mean[2])},
    "std": {"ate_m": float(std[0]), "rte_m": float(std[1]), "quat_mean_deg": float(std[2])},
    "A8q_ref": a8q,
    "GPS_vel": {"ate_m": float(ate_g), "rte_m": float(rte_g)},
    "EKF": {"ate_m": float(ate_e), "rte_m": float(rte_e), "quat_mean_deg": float(quat_e)},
}
json.dump(out, open("ablation_results.json", "w"), indent=2)
torch.save({"run_id": "ABLATION_LN_DROPOUT", "best_seed": best_overall[2], "state": best_overall[1],
            "feat_mean": feat_mean, "feat_std": feat_std, "y_mean": y_mean, "y_std": y_std,
            "dropout": DROPOUT, "W": W, "stride": STRIDE, "hop_sec": HOP_SEC}, "ablation_best.pth")
print("\nwrote ablation_results.json + ablation_best.pth  (A8q untouched: option_a_a8q.pth)")

seed 1: val best-quat ATE 1.660 quat 4.12  ->  TEST ATE 1.046 RTE 0.435 quat 4.66
seed 2: val best-quat ATE 2.436 quat 5.11  ->  TEST ATE 1.083 RTE 0.377 quat 5.76
seed 3: val best-quat ATE 2.328 quat 4.53  ->  TEST ATE 0.993 RTE 0.391 quat 5.05

==== ablation (channel LayerNorm + dropout) vs A8q ====
                         ATE     RTE    quat
A8q (frozen)            1.12    0.31    4.95
ablation mean           1.04    0.40    5.15
ablation std            0.04    0.02    0.46
GPS-vel                 1.10    0.45      --
insfilterMARG           0.78    0.20    7.46

wrote ablation_results.json + ablation_best.pth  (A8q untouched: option_a_a8q.pth)
